# SQL Mastery for Quantitative Research

## Professional SQL foundations for systematic trading workflows

This notebook is a **reference-and-recap notebook**, not an exercise sheet. Its objective is to make the SQL constructs required for quantitative research familiar enough that they can later be used independently in a TODO-based case study.

The focus is analytical SQL: retrieving, joining, validating, transforming, ranking, and aggregating market data before handing a research-ready dataset to pandas, NumPy, scikit-learn, or PyTorch.

### Learning objectives

By the end of the notebook, you should be comfortable with:

- relational tables, rows, columns, primary keys, and foreign keys;
- `SELECT`, aliases, expressions, `DISTINCT`, filtering, sorting, and limiting;
- `NULL` semantics and `COALESCE`;
- conditional logic with `CASE`;
- aggregation with `GROUP BY` and `HAVING`;
- joins and the duplicate-row risks they introduce;
- subqueries and common table expressions (CTEs);
- date extraction and date arithmetic;
- window functions, including `LAG`, `LEAD`, ranking, cumulative and rolling calculations;
- reshaping with conditional aggregation;
- set operations;
- data-quality checks;
- query organization and the division of work between SQL and pandas.

The examples use **DuckDB SQL** because it is lightweight and well suited to analytical workflows. The SQL concepts are largely portable to PostgreSQL and other relational databases, although some date and utility functions differ by dialect.


## 0. Setup

The notebook creates a small synthetic market database with four tables:

- `assets`: security metadata;
- `prices`: daily close prices and volumes;
- `signals`: daily model signals;
- `trades`: example executions.

The data is deliberately small enough to inspect while preserving the table structure used in real research databases.


In [1]:
# Run once if DuckDB is not installed:
# %pip install duckdb

import duckdb
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
con = duckdb.connect()

dates = pd.bdate_range("2024-01-02", periods=90)
assets = pd.DataFrame({
    "asset": ["EQ_US", "EQ_EU", "BOND_US", "BOND_EU", "GOLD", "OIL", "EURUSD", "USDJPY"],
    "asset_class": ["Equity", "Equity", "Rates", "Rates", "Commodity", "Commodity", "FX", "FX"],
    "region": ["US", "Europe", "US", "Europe", "Global", "Global", "Europe", "Japan"],
    "currency": ["USD", "EUR", "USD", "EUR", "USD", "USD", "USD", "JPY"],
})

rows = []
signal_rows = []
for j, asset in enumerate(assets["asset"]):
    rets = rng.normal(0.0002 + j * 0.00001, 0.008 + j * 0.0003, len(dates))
    close = (100 + 5 * j) * np.exp(np.cumsum(rets))
    volume = rng.integers(100_000, 2_000_000, len(dates))
    signal = pd.Series(rets).rolling(10, min_periods=3).mean().to_numpy() + rng.normal(0, 0.002, len(dates))
    rows.extend(zip(dates, [asset] * len(dates), close, volume))
    signal_rows.extend(zip(dates, [asset] * len(dates), signal))

prices = pd.DataFrame(rows, columns=["date", "asset", "close", "volume"])
signals = pd.DataFrame(signal_rows, columns=["date", "asset", "signal"])

trade_dates = dates[::9]
trades = pd.DataFrame({
    "trade_id": np.arange(1, len(trade_dates) * 3 + 1),
    "date": np.repeat(trade_dates, 3),
    "asset": np.tile(["EQ_US", "GOLD", "EURUSD"], len(trade_dates)),
    "quantity": rng.integers(-250, 251, len(trade_dates) * 3),
    "price": rng.normal(100, 8, len(trade_dates) * 3),
})

for df in [assets, prices, signals, trades]:
    string_cols = df.select_dtypes(include=["str", "string"]).columns
    df[string_cols] = df[string_cols].astype("object")

con.register("assets_df", assets)
con.register("prices_df", prices)
con.register("signals_df", signals)
con.register("trades_df", trades)

con.execute("CREATE OR REPLACE TABLE assets AS SELECT * FROM assets_df")
con.execute("CREATE OR REPLACE TABLE prices AS SELECT * FROM prices_df")
con.execute("CREATE OR REPLACE TABLE signals AS SELECT * FROM signals_df")
con.execute("CREATE OR REPLACE TABLE trades AS SELECT * FROM trades_df")


### Helper

`con.sql(...)` returns a DuckDB relation. Calling `.df()` converts the result to a pandas DataFrame. In a production workflow, SQL should normally reduce and shape the data **before** that conversion.


In [2]:
con.sql("SELECT * FROM assets ORDER BY asset").df()

# main functions:
# - con = duckdb.connect() -> used frequently
# - np.repeat(array, n_times) -> repeats each element n times [1, 2], 2 -> [1, 1, 2, 2]
# - np.tile(array, n_times) -> similar to repeat but placed at the end -> [1, 2, 1, 2]
# - con.sql('request').df() -> to get a pandas df at the end


,asset,asset_class,region,currency
0,BOND_EU,Rates,Europe,EUR
1,BOND_US,Rates,US,USD
2,EQ_EU,Equity,Europe,EUR
3,EQ_US,Equity,US,USD
4,EURUSD,FX,Europe,USD
5,GOLD,Commodity,Global,USD
6,OIL,Commodity,Global,USD
7,USDJPY,FX,Japan,JPY


## 1. Relational structure and keys

A relational database stores data in tables connected by keys.

For this notebook:

- `assets.asset` identifies an asset in the metadata table;
- `(prices.date, prices.asset)` should identify one price observation;
- `(signals.date, signals.asset)` should identify one signal observation;
- `trades.trade_id` identifies one execution.

A **primary key** uniquely identifies a row. A **foreign key** links a value to another table. Even when a research warehouse does not enforce these constraints physically, you should reason about the data as if the intended keys were explicit.

Before writing a complex query, ask:

1. What does one row represent?
2. Which columns uniquely identify a row?
3. What cardinality should a join have?


In [ ]:
con.sql('''
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT asset) AS n_assets,
    COUNT(DISTINCT date) AS n_dates
FROM prices
''').df()

# main functions
# con.sql('''SELECT COUNT(*) AS name FROM df''').df() -> count the nbr of rows, COUNT(DISTINCT col) similar to df.nunique().to_frame().T 


,n_rows,n_assets,n_dates
0,720,8,90


In [4]:
prices.nunique().to_frame().T


,date,asset,close,volume
0,90,8,720,720


## 2. SELECT, expressions, aliases, DISTINCT, ORDER BY, LIMIT

`SELECT` chooses columns and can compute new expressions. `AS` gives an output column a clear name.

SQL is declarative: you describe the result you want rather than iterating row by row.


In [5]:
con.sql('''
SELECT
    date,
    asset,
    close,
    volume,
    close * volume AS dollar_volume
FROM prices
ORDER BY date DESC, asset
LIMIT 10
''').df()


,date,asset,close,volume,dollar_volume
0,2024-05-06,BOND_EU,113.995642,1177181,1.341935e+08
1,2024-05-06,BOND_US,118.623899,1469322,1.742967e+08
2,2024-05-06,EQ_EU,110.748910,614016,6.800160e+07
3,2024-05-06,EQ_US,102.526799,1921262,1.969808e+08
4,2024-05-06,EURUSD,148.856645,310376,4.620153e+07
5,2024-05-06,GOLD,125.240365,1288408,1.613607e+08
6,2024-05-06,OIL,117.607452,1179075,1.386680e+08
7,2024-05-06,USDJPY,127.146277,151646,1.928122e+07
8,2024-05-03,BOND_EU,112.708879,1549906,1.746882e+08
9,2024-05-03,BOND_US,117.432862,1078649,1.266688e+08


In [6]:
test = (
    prices[['date', 'asset', 'close', 'volume']]
    .assign(dollar_volume = lambda x: x.close * x.volume)
    .sort_values(by=['date', 'asset'], ascending=[False, True])
    .head(10)
    .reset_index(drop=True)
)
test


,date,asset,close,volume,dollar_volume
0,2024-05-06,BOND_EU,113.995642,1177181,1.341935e+08
1,2024-05-06,BOND_US,118.623899,1469322,1.742967e+08
2,2024-05-06,EQ_EU,110.748910,614016,6.800160e+07
3,2024-05-06,EQ_US,102.526799,1921262,1.969808e+08
4,2024-05-06,EURUSD,148.856645,310376,4.620153e+07
5,2024-05-06,GOLD,125.240365,1288408,1.613607e+08
6,2024-05-06,OIL,117.607452,1179075,1.386680e+08
7,2024-05-06,USDJPY,127.146277,151646,1.928122e+07
8,2024-05-03,BOND_EU,112.708879,1549906,1.746882e+08
9,2024-05-03,BOND_US,117.432862,1078649,1.266688e+08


`DISTINCT` removes duplicate combinations from the selected columns. Use it intentionally; it should not be used to hide a join that accidentally duplicated rows.


In [ ]:
con.sql('''
SELECT DISTINCT asset
FROM prices
ORDER BY asset
''').df()

# main functions:
# con.sql('''
# SELECT cols 
# FROM df 
# ORDER BY col DESC 
# LIMIT size''').df() -> for multiple cols: col1, col2 not a list. DESC means descending order vs ASC
# Can use operations: SELECT close, volume, close * volume AS name 
# similar to:
# result = (
#     prices[['date', 'asset', 'close', 'volume']]
#     .assign(dollar_volume = lambda x: x.close * x.volume)
#     .sort_values(by=['date', 'asset'], ascending=[False, True])
#     .head(10)
#     .reset_index(drop=True)
# )
# 
# con.sql('''
# SELECT DISTINCT col 
# FROM df''').df() -> to drop duplicate, similar to df['col'].drop_duplicates().reset_index(drop=True).to_frame()


,asset
0,BOND_EU
1,BOND_US
2,EQ_EU
3,EQ_US
4,EURUSD
5,GOLD
6,OIL
7,USDJPY


In [8]:
test = prices['asset'].drop_duplicates().reset_index(drop=True).to_frame()
test


,asset
0,EQ_US
1,EQ_EU
2,BOND_US
3,BOND_EU
4,GOLD
5,OIL
6,EURUSD
7,USDJPY


## 3. Filtering: WHERE, IN, BETWEEN, LIKE

`WHERE` filters rows **before** aggregation.

Common predicates include:

- comparisons: `=`, `<>`, `>`, `>=`, `<`, `<=`;
- membership: `IN (...)`;
- ranges: `BETWEEN ... AND ...`;
- pattern matching: `LIKE`;
- boolean combinations: `AND`, `OR`, `NOT`.

Use parentheses when mixing `AND` and `OR`.


In [9]:
con.sql('''
SELECT date, asset, close
FROM prices
WHERE date BETWEEN DATE '2024-02-01' AND DATE '2024-02-29'
  AND asset IN ('EQ_US', 'EQ_EU')
ORDER BY date, asset
''').df()

# main functions:
# con.sql('''
# SELECT cols
# FROM df
# WHERE col BETWEEN ... AND ... 
# OR col IN ('str1', 'str2')''').df() -> conditions: WHERE + (operations (>, =, <=, ...), BETWEEN ... AND ..., BETWEEN DATE ... AND DATE ... if date, AND/OR/NOT to combine)
# 
# similar to:
# test = df[['date', 'asset', 'close']].sort_values(by=['date', 'asset'])
# test = test.loc[(test['date'] >= '2024-02-01') & (test['date'] <= '2024-02-29') & (test['asset'].isin(['EQ_US', 'EQ_EU']))].reset_index(drop=True)


,date,asset,close
0,2024-02-01,EQ_EU,101.261613
1,2024-02-01,EQ_US,100.218725
2,2024-02-02,EQ_EU,103.754699
3,2024-02-02,EQ_US,100.114928
4,2024-02-05,EQ_EU,102.772258
5,2024-02-05,EQ_US,99.792416
6,2024-02-06,EQ_EU,102.480136
7,2024-02-06,EQ_US,99.531594
8,2024-02-07,EQ_EU,102.792654
9,2024-02-07,EQ_US,99.976344


In [10]:
test = prices[['date', 'asset', 'close']].sort_values(by=['date', 'asset'])
test = test.loc[(test['date'] >= '2024-02-01') & (test['date'] <= '2024-02-29') & (test['asset'].isin(['EQ_US', 'EQ_EU']))].reset_index(drop=True)
test.head(5)


,date,asset,close
0,2024-02-01,EQ_EU,101.261613
1,2024-02-01,EQ_US,100.218725
2,2024-02-02,EQ_EU,103.754699
3,2024-02-02,EQ_US,100.114928
4,2024-02-05,EQ_EU,102.772258


## 4. NULL: missing information is not zero

SQL uses `NULL` for missing or unknown values. Comparisons such as `value = NULL` do not work because `NULL` is not an ordinary value.

Use:

- `IS NULL`;
- `IS NOT NULL`;
- `COALESCE(x, fallback)`.

Aggregations such as `AVG`, `SUM`, and `STDDEV_SAMP` generally ignore `NULL` observations.


In [11]:
con.sql('''
SELECT
    date,
    asset,
    signal,
    COALESCE(signal, 0.0) AS signal_filled
FROM signals
WHERE signal IS NULL
ORDER BY date, asset
LIMIT 10
''').df()

# main functions:
# con.sql('''
# SELECT cols
# FROM df
# WHERE col IS NULL).df() -> check nan with IS NULL and IS NOT NULL (condition with WHERE for example)
# replace within the SELECT: COALESCE(col, replacement) AS name -> fillna(replacement)
# 
# similar to signals[['date', 'asset', 'signal']].loc[signals['signal'].isna()].assign(signal_filled = lambda x: x.signal.fillna(0.0))


,date,asset,signal,signal_filled
0,2024-01-02,BOND_EU,NaN,0.0
1,2024-01-02,BOND_US,NaN,0.0
2,2024-01-02,EQ_EU,NaN,0.0
3,2024-01-02,EQ_US,NaN,0.0
4,2024-01-02,EURUSD,NaN,0.0
5,2024-01-02,GOLD,NaN,0.0
6,2024-01-02,OIL,NaN,0.0
7,2024-01-02,USDJPY,NaN,0.0
8,2024-01-03,BOND_EU,NaN,0.0
9,2024-01-03,BOND_US,NaN,0.0


In [12]:
test = signals[['date', 'asset', 'signal']].loc[signals['signal'].isna()].assign(signal_filled = lambda x: x.signal.fillna(0.0)).head(5)
test 


,date,asset,signal,signal_filled
0,2024-01-02,EQ_US,NaN,0.0
1,2024-01-03,EQ_US,NaN,0.0
90,2024-01-02,EQ_EU,NaN,0.0
91,2024-01-03,EQ_EU,NaN,0.0
180,2024-01-02,BOND_US,NaN,0.0


## 5. Conditional logic with CASE

`CASE` is SQL's general conditional expression. It is analogous to vectorized conditional logic such as `np.select` or chained boolean masks.

The result can be numeric, text, dates, or other compatible types.


In [ ]:
con.sql('''
SELECT
    date,
    asset,
    signal,
    CASE
        WHEN signal > 0.002 THEN 'Positive'
        WHEN signal < -0.002 THEN 'Negative'
        ELSE 'Neutral'
    END AS signal_bucket
FROM signals
WHERE signal IS NOT NULL
ORDER BY date, asset
LIMIT 12
''').df()

# main functions:
# con.sql('''
# SELECT col 
#    CASE 
#       WHEN cdt THEN result
#       ELSE result
#    END AS name 
# FROM df''').df() -> like np.select(conditions, results)
# 
# similar to:
# test = (signals[['date', 'asset', 'signal']]
#         .dropna(axis=0)
#         .sort_values(by=['date', 'asset'])
#         .assign(signal_bucket = lambda x: np.select([x.signal > 0.002, x.signal < -0.002], ['Positive', 'Negative'], default='Neutral'))
#         .reset_index(drop=True)
#         )


,date,asset,signal,signal_bucket
0,2024-01-04,BOND_EU,-0.005625,Negative
1,2024-01-04,BOND_US,0.001994,Neutral
2,2024-01-04,EQ_EU,0.012694,Positive
3,2024-01-04,EQ_US,0.001917,Neutral
4,2024-01-04,EURUSD,0.000013,Neutral
5,2024-01-04,GOLD,-0.002603,Negative
6,2024-01-04,OIL,-0.002896,Negative
7,2024-01-04,USDJPY,-0.001788,Neutral
8,2024-01-05,BOND_EU,-0.001388,Neutral
9,2024-01-05,BOND_US,0.005337,Positive


In [14]:
test = (signals[['date', 'asset', 'signal']]
        .dropna(axis=0)
        .sort_values(by=['date', 'asset'])
        .assign(signal_bucket = lambda x: np.select([x.signal > 0.002, x.signal < -0.002], ['Positive', 'Negative'], default='Neutral'))
        .reset_index(drop=True)
        )
test.head(5)


,date,asset,signal,signal_bucket
0,2024-01-04,BOND_EU,-0.005625,Negative
1,2024-01-04,BOND_US,0.001994,Neutral
2,2024-01-04,EQ_EU,0.012694,Positive
3,2024-01-04,EQ_US,0.001917,Neutral
4,2024-01-04,EURUSD,0.000013,Neutral


## 6. Aggregation: GROUP BY

Aggregate functions collapse multiple rows into summary values.

Core functions:

- `COUNT(*)`;
- `COUNT(column)`;
- `COUNT(DISTINCT column)`;
- `SUM`;
- `AVG`;
- `MIN`, `MAX`;
- `STDDEV_SAMP`.

Every selected column that is not aggregated must normally appear in `GROUP BY`.


In [15]:
con.sql('''
SELECT
    asset,
    AVG(close) AS avg_close,
    STDDEV_SAMP(close) AS close_std,
    AVG(volume) AS avg_volume,
    COUNT(*) AS n_obs
FROM prices
GROUP BY asset
ORDER BY asset
''').df()

# main functions:
# con.sql('''
# SELECT 
#    AGG_FCT(col) AS name
# FROM df
# GROUP BY col''') -> aggregate functions such as: COUNT(*), COUNT(col), COUNT(DISTINCT col), SUM, AVG, MIN/MAX, STDDEV_SAMP
# 
# similar to:
# test = (prices[['asset', 'close', 'volume']]
#         .groupby('asset')
#         .agg(
#             avg_close=('close','mean'),
#             close_std=('close','std'),
#             avg_volume=('volume','mean'),
#             n_obs=('close','count')
#         )
#         .sort_values(by='asset')
#         )


,asset,avg_close,close_std,avg_volume,n_obs
0,BOND_EU,112.660750,4.603724,1.013987e+06,90
1,BOND_US,115.476783,2.534195,1.117218e+06,90
2,EQ_EU,104.912675,2.200180,1.127521e+06,90
3,EQ_US,101.965246,1.974401,1.001868e+06,90
4,EURUSD,136.086343,5.262179,1.102091e+06,90
5,GOLD,123.959129,3.734945,1.132150e+06,90
6,OIL,119.892677,3.802344,1.098402e+06,90
7,USDJPY,128.854377,3.528672,1.099388e+06,90


In [16]:
test = (prices[['asset', 'close', 'volume']]
        .groupby('asset')
        .agg(
            avg_close=('close','mean'),
            close_std=('close','std'),
            avg_volume=('volume','mean'),
            n_obs=('close','count')
        )
        .sort_values(by='asset')
        )
test.head(5)


,avg_close,close_std,avg_volume,n_obs
asset,,,,
BOND_EU,112.660750,4.603724,1.013987e+06,90
BOND_US,115.476783,2.534195,1.117218e+06,90
EQ_EU,104.912675,2.200180,1.127521e+06,90
EQ_US,101.965246,1.974401,1.001868e+06,90
EURUSD,136.086343,5.262179,1.102091e+06,90


## 7. WHERE versus HAVING

The distinction is fundamental:

- `WHERE` filters **rows before grouping**;
- `HAVING` filters **groups after aggregation**.

If the condition depends on an aggregate such as `AVG(...)` or `COUNT(...)`, it belongs in `HAVING`.


In [17]:
con.sql('''
SELECT
    asset,
    AVG(volume) AS avg_volume
FROM prices
WHERE date >= DATE '2024-02-01'
GROUP BY asset
HAVING AVG(volume) > 900000
ORDER BY avg_volume DESC
''').df()

# main functions:
# - WHERE for condition on initial column vs HAVING for condition on aggregated column


,asset,avg_volume
0,OIL,1.144554e+06
1,EQ_EU,1.142882e+06
2,BOND_US,1.114117e+06
3,GOLD,1.105207e+06
4,EURUSD,1.103920e+06
5,USDJPY,1.094048e+06
6,EQ_US,1.005465e+06
7,BOND_EU,9.981712e+05


## 8. Joins

Joins combine tables through matching keys.

The main forms you need are:

- `INNER JOIN`: keep matched rows only;
- `LEFT JOIN`: keep every row from the left table and attach matches from the right;
- `FULL OUTER JOIN`: retain unmatched rows from both sides.

For research work, the most important issue is often not syntax but **join cardinality**.


In [18]:
con.sql('''
SELECT
    p.date,
    p.asset,
    p.close,
    a.asset_class,
    a.region
FROM prices AS p
INNER JOIN assets AS a
    ON p.asset = a.asset
ORDER BY p.date, p.asset
LIMIT 12
''').df()


,date,asset,close,asset_class,region
0,2024-01-02,BOND_EU,115.305626,Rates,Europe
1,2024-01-02,BOND_US,111.083910,Rates,US
2,2024-01-02,EQ_EU,105.221184,Equity,Europe
3,2024-01-02,EQ_US,100.264122,Equity,US
4,2024-01-02,EURUSD,130.594429,FX,Europe
5,2024-01-02,GOLD,119.284618,Commodity,Global
6,2024-01-02,OIL,123.862463,Commodity,Global
7,2024-01-02,USDJPY,135.257664,FX,Japan
8,2024-01-03,BOND_EU,114.601711,Rates,Europe
9,2024-01-03,BOND_US,111.475285,Rates,US


### Join cardinality and accidental duplication

Suppose `(date, asset)` is unique in `prices` and `signals`. Joining on both columns should preserve one row per price observation.

Joining only on `asset` would create many-to-many matches across dates and explode the row count.

A useful professional habit is to check row counts and key uniqueness before and after important joins.


In [19]:
con.sql('''
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT (date, asset)) AS n_unique_date_asset
FROM (
    SELECT p.date, p.asset, p.close, s.signal
    FROM prices AS p
    LEFT JOIN signals AS s
      ON p.date = s.date
     AND p.asset = s.asset
)
''').df()

# main functions:
# con.sql('''
# SELECT
#    p.col,
#    a.col,
# FROM df1 as p
# INNER JOIN df2 as a
#    ON p.sharedcol = a.sharedcol''').df() -> to merge df, INNER JOIN/LEFT JOIN/FULL OUTER JOIN similar to how='' with pandas merge 
# 
# Remark: we need to confirm uniqueness when joining, in our sample with asset and dates, uniqueness happens only if we combine (asset, date) 
# -> we need to join using both
# LEFT JOIN on df2 AS s
#    ON p.date = s.date
#    AND p.asset = s.asset


,n_rows,n_unique_date_asset
0,720,720


## 9. Subqueries

A subquery is a query nested inside another query. It is useful when an intermediate result is needed only once.

The example below selects observations whose volume is above that asset's full-sample average.


In [ ]:
con.sql('''
SELECT p.date, p.asset, p.volume
FROM prices AS p
WHERE p.volume > (
    SELECT AVG(p2.volume)
    FROM prices AS p2
    WHERE p2.asset = p.asset
)
ORDER BY p.asset, p.date
LIMIT 15
''').df()

# main functions:
# WHERE p.volume > (
#    SELECT AVG(p2.volume)
#    FROM df AS p2
#    WHERE p2.asset = p.asset) -> () is a subquery, we need its result inside the main query for our condition. 
# Remark: with this WHERE, it computes volume per asset. Would need WHERE ... > (... WHERE p2.asset IN (SELECT DISTINCT p1.asset FROM df AS p1))
# 
# would usually do in 2 lines with pandas: 
# avg_vol = prices.groupby('asset')['volume'].transform('mean')
# test = (
#     prices[['date','asset','volume']]
#     .loc[prices['volume'] > avg_vol]
#     .sort_values(by=['asset','date'])
#     .reset_index(drop=True)
#     )


,date,asset,volume
0,2024-01-02,BOND_EU,1079258
1,2024-01-04,BOND_EU,1307851
2,2024-01-08,BOND_EU,1664224
3,2024-01-10,BOND_EU,1737570
4,2024-01-12,BOND_EU,1498201
5,2024-01-18,BOND_EU,1433990
6,2024-01-22,BOND_EU,1352397
7,2024-01-23,BOND_EU,1477848
8,2024-01-24,BOND_EU,1705953
9,2024-01-25,BOND_EU,1445373


In [21]:
avg_vol = prices.groupby('asset')['volume'].transform('mean')
test = (
    prices[['date','asset','volume']]
    .loc[prices['volume'] > avg_vol]
    .sort_values(by=['asset','date'])
    .reset_index(drop=True)
    )
test.head(5)

,date,asset,volume
0,2024-01-02,BOND_EU,1079258
1,2024-01-04,BOND_EU,1307851
2,2024-01-08,BOND_EU,1664224
3,2024-01-10,BOND_EU,1737570
4,2024-01-12,BOND_EU,1498201


## 10. Common table expressions (WITH)

CTEs give names to intermediate query steps. For analytical work, they are often clearer than deeply nested subqueries.

Think of each CTE as a well-defined transformation stage. This is close to assigning intermediate DataFrames in pandas, but the database can optimize the complete query plan.


In [ ]:
con.sql('''
WITH price_returns AS (
    SELECT
        date,
        asset,
        close / LAG(close) OVER (
            PARTITION BY asset
            ORDER BY date
        ) - 1 AS ret
    FROM prices
),
asset_stats AS (
    SELECT
        asset,
        AVG(ret) AS mean_ret,
        STDDEV_SAMP(ret) AS vol
    FROM price_returns
    GROUP BY asset
)
SELECT *
FROM asset_stats
ORDER BY vol DESC
''').df()

# main functions:
# con.sql('''
# WITH intermediate_name AS (
# query
# ),
# main query''').df() -> names intermediate tables for clarity and syntax


,asset,mean_ret,vol
0,USDJPY,-0.000642,0.010315
1,OIL,-0.000532,0.010025
2,EURUSD,0.001521,0.009944
3,GOLD,0.000591,0.009380
4,BOND_EU,-0.000087,0.009173
5,EQ_EU,0.000616,0.009054
6,BOND_US,0.000778,0.009031
7,EQ_US,0.000269,0.006064


## 11. Dates

Dates are first-class SQL values. Useful operations include:

- typed literals such as `DATE '2024-01-01'`;
- `EXTRACT(...)`;
- `DATE_TRUNC(...)`;
- interval arithmetic.

Exact date syntax varies more across SQL dialects than basic `SELECT`/`JOIN` syntax.


In [ ]:
con.sql('''
SELECT
    date,
    EXTRACT(YEAR FROM date) AS year,
    EXTRACT(MONTH FROM date) AS month,
    DATE_TRUNC('month', date) AS month_start
FROM prices
GROUP BY date
ORDER BY date
LIMIT 10
''').df()

# main functions:


,date,year,month,month_start
0,2024-01-02,2024,1,2024-01-01
1,2024-01-03,2024,1,2024-01-01
2,2024-01-04,2024,1,2024-01-01
3,2024-01-05,2024,1,2024-01-01
4,2024-01-08,2024,1,2024-01-01
5,2024-01-09,2024,1,2024-01-01
6,2024-01-10,2024,1,2024-01-01
7,2024-01-11,2024,1,2024-01-01
8,2024-01-12,2024,1,2024-01-01
9,2024-01-15,2024,1,2024-01-01


## 12. Window functions: the analytical SQL core

Window functions calculate across related rows **without collapsing them**.

General form:

```sql
FUNCTION(...) OVER (
    PARTITION BY ...
    ORDER BY ...
    ROWS BETWEEN ...
)
```

`PARTITION BY` defines independent groups. `ORDER BY` defines sequence within each group.

This family is especially important for market data.


### 12.1 LAG and LEAD

`LAG(x)` accesses a previous row within the ordered partition. `LEAD(x)` accesses a subsequent row.

For prices, `LAG` is the natural SQL analogue of a grouped `shift(1)`.


In [24]:
con.sql('''
SELECT
    date,
    asset,
    close,
    LAG(close) OVER (
        PARTITION BY asset
        ORDER BY date
    ) AS previous_close,
    close / LAG(close) OVER (
        PARTITION BY asset
        ORDER BY date
    ) - 1 AS simple_return
FROM prices
ORDER BY asset, date
LIMIT 15
''').df()


,date,asset,close,previous_close,simple_return
0,2024-01-02,BOND_EU,115.305626,NaN,NaN
1,2024-01-03,BOND_EU,114.601711,115.305626,-0.006105
2,2024-01-04,BOND_EU,113.292497,114.601711,-0.011424
3,2024-01-05,BOND_EU,114.164642,113.292497,0.007698
4,2024-01-08,BOND_EU,114.546500,114.164642,0.003345
5,2024-01-09,BOND_EU,117.028327,114.546500,0.021667
6,2024-01-10,BOND_EU,117.493815,117.028327,0.003978
7,2024-01-11,BOND_EU,117.927055,117.493815,0.003687
8,2024-01-12,BOND_EU,117.779071,117.927055,-0.001255
9,2024-01-15,BOND_EU,118.665653,117.779071,0.007527


### 12.2 Cross-sectional ranking

Ranking functions include:

- `ROW_NUMBER`: unique sequential positions;
- `RANK`: ties share a rank and leave gaps;
- `DENSE_RANK`: ties share a rank without gaps;
- `NTILE(n)`: approximately splits ordered observations into `n` buckets.

For cross-sectional signals, the partition is often the date.


In [25]:
con.sql('''
SELECT
    date,
    asset,
    signal,
    RANK() OVER (
        PARTITION BY date
        ORDER BY signal
    ) AS signal_rank,
    NTILE(4) OVER (
        PARTITION BY date
        ORDER BY signal
    ) AS signal_quartile
FROM signals
WHERE signal IS NOT NULL
ORDER BY date, signal_rank
LIMIT 20
''').df()


,date,asset,signal,signal_rank,signal_quartile
0,2024-01-04,BOND_EU,-0.005625,1,1
1,2024-01-04,OIL,-0.002896,2,1
2,2024-01-04,GOLD,-0.002603,3,2
3,2024-01-04,USDJPY,-0.001788,4,2
4,2024-01-04,EURUSD,0.000013,5,3
5,2024-01-04,EQ_US,0.001917,6,3
6,2024-01-04,BOND_US,0.001994,7,4
7,2024-01-04,EQ_EU,0.012694,8,4
8,2024-01-05,GOLD,-0.006729,1,1
9,2024-01-05,USDJPY,-0.006073,2,1


### 12.3 Cumulative calculations

A windowed aggregate can preserve every row while calculating a cumulative statistic.

Explicitly writing the frame is good practice because default window frames can differ in ways that matter.


In [26]:
con.sql('''
WITH returns AS (
    SELECT
        date,
        asset,
        close / LAG(close) OVER (
            PARTITION BY asset
            ORDER BY date
        ) - 1 AS ret
    FROM prices
)
SELECT
    date,
    asset,
    ret,
    SUM(ret) OVER (
        PARTITION BY asset
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_arithmetic_return
FROM returns
ORDER BY asset, date
LIMIT 15
''').df()


,date,asset,ret,cumulative_arithmetic_return
0,2024-01-02,BOND_EU,NaN,NaN
1,2024-01-03,BOND_EU,-0.006105,-0.006105
2,2024-01-04,BOND_EU,-0.011424,-0.017529
3,2024-01-05,BOND_EU,0.007698,-0.009831
4,2024-01-08,BOND_EU,0.003345,-0.006486
5,2024-01-09,BOND_EU,0.021667,0.015181
6,2024-01-10,BOND_EU,0.003978,0.019158
7,2024-01-11,BOND_EU,0.003687,0.022846
8,2024-01-12,BOND_EU,-0.001255,0.021591
9,2024-01-15,BOND_EU,0.007527,0.029118


### 12.4 Rolling windows

A fixed trailing window uses a bounded frame. The example below computes a 20-observation rolling sample volatility.

`ROWS BETWEEN 19 PRECEDING AND CURRENT ROW` means at most 20 rows including the current row.


In [27]:
con.sql('''
WITH returns AS (
    SELECT
        date,
        asset,
        close / LAG(close) OVER (
            PARTITION BY asset
            ORDER BY date
        ) - 1 AS ret
    FROM prices
)
SELECT
    date,
    asset,
    ret,
    STDDEV_SAMP(ret) OVER (
        PARTITION BY asset
        ORDER BY date
        ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
    ) AS rolling_vol_20
FROM returns
ORDER BY asset, date
LIMIT 30
''').df()


,date,asset,ret,rolling_vol_20
0,2024-01-02,BOND_EU,NaN,NaN
1,2024-01-03,BOND_EU,-0.006105,NaN
2,2024-01-04,BOND_EU,-0.011424,0.003761
3,2024-01-05,BOND_EU,0.007698,0.009870
4,2024-01-08,BOND_EU,0.003345,0.008712
5,2024-01-09,BOND_EU,0.021667,0.012861
6,2024-01-10,BOND_EU,0.003978,0.011509
7,2024-01-11,BOND_EU,0.003687,0.010508
8,2024-01-12,BOND_EU,-0.001255,0.009859
9,2024-01-15,BOND_EU,0.007527,0.009362


### 12.5 Expanding calculations

An expanding window starts at the beginning of the partition:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

If today's classification must exclude today's observation, end the frame at `1 PRECEDING`.


In [28]:
con.sql('''
WITH returns AS (
    SELECT
        date,
        asset,
        close / LAG(close) OVER (
            PARTITION BY asset
            ORDER BY date
        ) - 1 AS ret
    FROM prices
)
SELECT
    date,
    asset,
    ret,
    AVG(ret) OVER (
        PARTITION BY asset
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS historical_mean_ex_today
FROM returns
ORDER BY asset, date
LIMIT 20
''').df()


,date,asset,ret,historical_mean_ex_today
0,2024-01-02,BOND_EU,NaN,NaN
1,2024-01-03,BOND_EU,-0.006105,NaN
2,2024-01-04,BOND_EU,-0.011424,-0.006105
3,2024-01-05,BOND_EU,0.007698,-0.008764
4,2024-01-08,BOND_EU,0.003345,-0.003277
5,2024-01-09,BOND_EU,0.021667,-0.001621
6,2024-01-10,BOND_EU,0.003978,0.003036
7,2024-01-11,BOND_EU,0.003687,0.003193
8,2024-01-12,BOND_EU,-0.001255,0.003264
9,2024-01-15,BOND_EU,0.007527,0.002699


## 13. QUALIFY: filtering after window functions

DuckDB supports `QUALIFY`, which filters rows after window functions have been evaluated.

This is useful for tasks such as "top 2 signals per date" without adding another CTE. PostgreSQL does not currently use `QUALIFY` in the same way, so a subquery/CTE is the more portable pattern.


In [29]:
con.sql('''
SELECT
    date,
    asset,
    signal,
    RANK() OVER (
        PARTITION BY date
        ORDER BY signal DESC
    ) AS rank_desc
FROM signals
WHERE signal IS NOT NULL
QUALIFY rank_desc <= 2
ORDER BY date, rank_desc
LIMIT 20
''').df()


,date,asset,signal,rank_desc
0,2024-01-04,EQ_EU,0.012694,1
1,2024-01-04,BOND_US,0.001994,2
2,2024-01-05,EQ_EU,0.008798,1
3,2024-01-05,EQ_US,0.006105,2
4,2024-01-08,BOND_US,0.010521,1
5,2024-01-08,EQ_EU,0.007990,2
6,2024-01-09,BOND_US,0.006099,1
7,2024-01-09,USDJPY,0.003152,2
8,2024-01-10,BOND_US,0.005687,1
9,2024-01-10,BOND_EU,0.003571,2


## 14. Conditional aggregation and pivot-style output

SQL often reshapes categories into columns through conditional aggregation.

This is particularly useful for compact reporting tables.


In [30]:
con.sql('''
SELECT
    date,
    AVG(CASE WHEN a.asset_class = 'Equity' THEN p.close END) AS equity_avg_close,
    AVG(CASE WHEN a.asset_class = 'Rates' THEN p.close END) AS rates_avg_close,
    AVG(CASE WHEN a.asset_class = 'Commodity' THEN p.close END) AS commodity_avg_close,
    AVG(CASE WHEN a.asset_class = 'FX' THEN p.close END) AS fx_avg_close
FROM prices AS p
JOIN assets AS a
  ON p.asset = a.asset
GROUP BY date
ORDER BY date
LIMIT 10
''').df()


,date,equity_avg_close,rates_avg_close,commodity_avg_close,fx_avg_close
0,2024-01-02,102.742653,113.194768,121.573540,132.926047
1,2024-01-03,103.459383,113.038498,121.397514,133.125427
2,2024-01-04,104.623881,112.333316,121.690501,131.422489
3,2024-01-05,104.638045,112.948961,120.444170,132.947802
4,2024-01-08,103.749249,114.097668,120.546020,133.045332
5,2024-01-09,102.602621,116.375428,120.523297,132.839620
6,2024-01-10,102.412487,116.655445,120.315032,133.457718
7,2024-01-11,102.448722,116.964656,122.254258,132.441988
8,2024-01-12,102.999948,117.442876,121.884651,131.473313
9,2024-01-15,102.361836,117.474661,122.907312,131.892462


## 15. Set operations

Set operations combine compatible query results:

- `UNION` combines and removes duplicates;
- `UNION ALL` combines and preserves duplicates;
- `INTERSECT` keeps common rows;
- `EXCEPT` keeps rows in the first result but not the second.

`UNION ALL` is usually preferable when duplicate removal is not required because it avoids unnecessary work.


In [31]:
con.sql('''
SELECT asset FROM assets WHERE asset_class = 'Equity'
UNION ALL
SELECT asset FROM assets WHERE asset_class = 'Commodity'
ORDER BY asset
''').df()


,asset
0,EQ_EU
1,EQ_US
2,GOLD
3,OIL


## 16. Data-quality checks

A researcher should be able to validate the data before trusting downstream results.

Typical checks include:

- duplicate intended keys;
- missing values;
- invalid ranges;
- unexpected category values;
- gaps in coverage;
- join mismatches.

These checks are often cheap to run in SQL before downloading the dataset.


In [32]:
# Duplicate intended keys
con.sql('''
SELECT date, asset, COUNT(*) AS n
FROM prices
GROUP BY date, asset
HAVING COUNT(*) > 1
ORDER BY n DESC
''').df()


,date,asset,n


In [33]:
# Coverage by asset
con.sql('''
SELECT
    asset,
    MIN(date) AS first_date,
    MAX(date) AS last_date,
    COUNT(*) AS n_obs,
    COUNT(close) AS n_non_null_close
FROM prices
GROUP BY asset
ORDER BY asset
''').df()


,asset,first_date,last_date,n_obs,n_non_null_close
0,BOND_EU,2024-01-02,2024-05-06,90,90
1,BOND_US,2024-01-02,2024-05-06,90,90
2,EQ_EU,2024-01-02,2024-05-06,90,90
3,EQ_US,2024-01-02,2024-05-06,90,90
4,EURUSD,2024-01-02,2024-05-06,90,90
5,GOLD,2024-01-02,2024-05-06,90,90
6,OIL,2024-01-02,2024-05-06,90,90
7,USDJPY,2024-01-02,2024-05-06,90,90


## 17. Query execution order: the mental model

A useful conceptual order is:

1. `FROM` / `JOIN`
2. `WHERE`
3. `GROUP BY`
4. aggregate calculations
5. `HAVING`
6. window functions
7. `SELECT`
8. `QUALIFY` where supported
9. `DISTINCT`
10. `ORDER BY`
11. `LIMIT`

The database optimizer may physically execute operations differently, but this logical model explains many syntax restrictions.

For example, a window result generally cannot be filtered directly in `WHERE` because `WHERE` conceptually happens earlier.


## 18. Professional query style

Readable SQL matters because research queries are reviewed, reused, and debugged.

Recommended conventions:

- uppercase SQL keywords consistently;
- use descriptive table aliases (`p`, `s`, `a`) rather than arbitrary letters;
- qualify ambiguous columns;
- one selected expression per line in nontrivial queries;
- put each join condition on a separate logical line;
- use CTEs to name meaningful transformation stages;
- avoid `SELECT *` in durable production/research queries;
- document assumptions rather than obvious syntax;
- verify key uniqueness instead of using `DISTINCT` as a repair mechanism.

Example:


In [34]:
query = '''
WITH price_returns AS (
    SELECT
        p.date,
        p.asset,
        a.asset_class,
        p.close / LAG(p.close) OVER (
            PARTITION BY p.asset
            ORDER BY p.date
        ) - 1 AS ret
    FROM prices AS p
    INNER JOIN assets AS a
        ON p.asset = a.asset
),
class_summary AS (
    SELECT
        date,
        asset_class,
        AVG(ret) AS equal_weight_return,
        COUNT(ret) AS n_assets
    FROM price_returns
    GROUP BY date, asset_class
)
SELECT
    date,
    asset_class,
    equal_weight_return,
    n_assets
FROM class_summary
WHERE n_assets > 0
ORDER BY date, asset_class
'''

con.sql(query).df().head(12)


,date,asset_class,equal_weight_return,n_assets
0,2024-01-03,Commodity,-0.001563,2
1,2024-01-03,Equity,0.006621,2
2,2024-01-03,FX,0.001390,2
3,2024-01-03,Rates,-0.001291,2
4,2024-01-04,Commodity,0.002468,2
5,2024-01-04,Equity,0.011068,2
6,2024-01-04,FX,-0.012723,2
7,2024-01-04,Rates,-0.006166,2
8,2024-01-05,Commodity,-0.010321,2
9,2024-01-05,Equity,0.000453,2


# Functions recap

---

* `con = duckdb.connect()` -> used frequently
* `np.repeat(array, n_times)` -> repeats each element n times `[1, 2], 2 -> [1, 1, 2, 2]`
* `np.tile(array, n_times)` -> similar to repeat but placed at the end -> `[1, 2, 1, 2]`
* `con.sql('request').df()` -> to get a pandas df at the end

---

```python
con.sql('''SELECT COUNT(*) AS name FROM df''').df()
```

-> count the nbr of rows, `COUNT(DISTINCT col)` similar to `df.nunique().to_frame().T`

---

```python
con.sql('''
SELECT cols
FROM df
ORDER BY col DESC
LIMIT size''').df()
```

-> for multiple cols: `col1, col2` not a list. DESC means descending order vs ASC

Can use operations: `SELECT close, volume, close * volume AS name`

similar to:

```python
result = (
    prices[['date', 'asset', 'close', 'volume']]
    .assign(dollar_volume = lambda x: x.close * x.volume)
    .sort_values(by=['date', 'asset'], ascending=[False, True])
    .head(10)
    .reset_index(drop=True)
)
```

```python
con.sql('''
SELECT DISTINCT col
FROM df''').df()
```

-> to drop duplicate, similar to `df['col'].drop_duplicates().reset_index(drop=True).to_frame()`

---

```python
con.sql('''
SELECT cols
FROM df
WHERE col BETWEEN ... AND ...
OR col IN ('str1', 'str2')''').df()
```

-> conditions: WHERE + (operations (`>`, `=`, `<=`, `...`), `BETWEEN ... AND ...`, `BETWEEN DATE ... AND DATE ...` if date, `AND/OR/NOT` to combine)

similar to:

```python
test = df[['date', 'asset', 'close']].sort_values(by=['date', 'asset'])
test = test.loc[
    (test['date'] >= '2024-02-01')
    & (test['date'] <= '2024-02-29')
    & (test['asset'].isin(['EQ_US', 'EQ_EU']))
].reset_index(drop=True)
```

```python
con.sql('''
SELECT cols
FROM df
WHERE col IS NULL).df()
```

-> check nan with `IS NULL` and `IS NOT NULL` (condition with WHERE for example)

replace within the SELECT: `COALESCE(col, replacement) AS name` -> `fillna(replacement)`

similar to:

```python
signals[['date', 'asset', 'signal']].loc[signals['signal'].isna()].assign(signal_filled = lambda x: x.signal.fillna(0.0))
```

---

```python
con.sql('''
SELECT col 
   CASE 
      WHEN cdt THEN result
      ELSE result
   END AS name 
FROM df''').df()
```

-> like np.select(conditions, results)

similar to:

```python
test = (signals[['date', 'asset', 'signal']]
        .dropna(axis=0)
        .sort_values(by=['date', 'asset'])
        .assign(signal_bucket = lambda x: np.select([x.signal > 0.002, x.signal < -0.002], ['Positive', 'Negative'], default='Neutral'))
        .reset_index(drop=True)
        )
```

---

```python
con.sql('''
SELECT 
   AGG_FCT(col) AS name
FROM df
GROUP BY col''') 
```

-> aggregate functions such as: `COUNT(*)`, `COUNT(col)`, `COUNT(DISTINCT col)`, `SUM`, `AVG`, `MIN/MAX`, `STDDEV_SAMP`

similar to:
```python
test = (prices[['asset', 'close', 'volume']]
        .groupby('asset')
        .agg(
            avg_close=('close','mean'),
            close_std=('close','std'),
            avg_volume=('volume','mean'),
            n_obs=('close','count')
        )
        .sort_values(by='asset')
        )
```

---

`WHERE` for condition on initial column vs `HAVING` for condition on aggregated column

---

```python
con.sql('''
SELECT
   p.col,
   a.col,
FROM df1 as p
INNER JOIN df2 as a
   ON p.sharedcol = a.sharedcol''').df() 
```
   
-> to merge df, `INNER JOIN`/`LEFT JOIN`/`FULL OUTER JOIN` similar to how='' with pandas merge 

`Remark`: we need to confirm uniqueness when joining, in our sample with asset and dates, uniqueness happens only if we combine (asset, date) 
-> we need to join using both
```python
LEFT JOIN on df2 AS s
   ON p.date = s.date
   AND p.asset = s.asset
```

---

```python
WHERE p.volume > (
   SELECT AVG(p2.volume)
   FROM df AS p2
   WHERE p2.asset = p.asset)  
```

-> () is a subquery, we need its result inside the main query for our condition.
`Remark`: with this WHERE, it computes volume per asset. 

Would need WHERE ... > (... WHERE p2.asset IN (SELECT DISTINCT p1.asset FROM df AS p1))

would usually do in 2 lines with pandas: 

```python
avg_vol = prices.groupby('asset')['volume'].transform('mean')
test = (
    prices[['date','asset','volume']]
    .loc[prices['volume'] > avg_vol]
    .sort_values(by=['asset','date'])
    .reset_index(drop=True)
    )
```

---

```python
con.sql('''
WITH intermediate_name AS (
query
),
main query''').df() 
```

-> names intermediate tables for clarity and syntax

---

